In [1]:
import pandas as pd
import numpy as np

test_df = pd.read_csv('data/original_data/ais_test.csv')
train_df = pd.read_csv('data/original_data/ais_train.csv', sep='|')
schedules_df = pd.read_csv('data/original_data/schedules_to_may_2024.csv', sep='|')
ports_df = pd.read_csv("data/original_data/ports.csv", sep='|')

train_df.head()


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,portId
0,2024-01-01 00:00:25,284.0,0.7,0,88,0,01-09 23:00,-34.74370,-57.85130,61e9f3a8b937134a3c4bfdf7,61d371c43aeaecc07011a37f
1,2024-01-01 00:00:36,109.6,0.0,-6,347,1,12-29 20:00,8.89440,-79.47939,61e9f3d4b937134a3c4bff1f,634c4de270937fc01c3a7689
2,2024-01-01 00:01:45,111.0,11.0,0,112,0,01-02 09:00,39.19065,-76.47567,61e9f436b937134a3c4c0131,61d3847bb7b7526e1adf3d19
3,2024-01-01 00:03:11,96.4,0.0,0,142,1,12-31 20:00,-34.41189,151.02067,61e9f3b4b937134a3c4bfe77,61d36f770a1807568ff9a126
4,2024-01-01 00:03:51,214.0,19.7,0,215,0,01-25 12:00,35.88379,-5.91636,61e9f41bb937134a3c4c0087,634c4de270937fc01c3a74f3


#### LAST KNOWN POSITION FEATURES

In [2]:
import pandas as pd

train_df = train_df.sort_values(by=['vesselId', 'time']).reset_index(drop=True)

# Create a mask for the first occurrence of each vesselId
first_occurrences = ~train_df['vesselId'].duplicated()

# Calculate the shifted values for steps 1-5
for steps in range(1, 6):
    train_df[f'latitude_{steps}_steps_ago'] = train_df.groupby('vesselId')['latitude'].shift(steps)
    train_df[f'longitude_{steps}_steps_ago'] = train_df.groupby('vesselId')['longitude'].shift(steps)
    train_df[f'time_position_{steps}_steps_ago'] = train_df.groupby('vesselId')['time'].shift(steps)
    
    # Set the values to NaN for the first occurrence of each vesselId
    train_df.loc[first_occurrences, f'latitude_{steps}_steps_ago'] = np.nan
    train_df.loc[first_occurrences, f'longitude_{steps}_steps_ago'] = np.nan
    train_df.loc[first_occurrences, f'time_position_{steps}_steps_ago'] = np.nan

# Calculate max and min changes in last 5 steps
def calculate_max_min_changes(group):
    # Create arrays of previous values including current value
    lat_values = np.array([
        group['latitude'],
        group['latitude_1_steps_ago'],
        group['latitude_2_steps_ago'],
        group['latitude_3_steps_ago'],
        group['latitude_4_steps_ago'],
        group['latitude_5_steps_ago']
    ])
    
    long_values = np.array([
        group['longitude'],
        group['longitude_1_steps_ago'],
        group['longitude_2_steps_ago'],
        group['longitude_3_steps_ago'],
        group['longitude_4_steps_ago'],
        group['longitude_5_steps_ago']
    ])
    
    # Calculate changes between consecutive values
    lat_changes = np.diff(lat_values)
    long_changes = np.diff(long_values)
    
    # Handle cases with NaN values
    lat_changes = lat_changes[~np.isnan(lat_changes)]
    long_changes = long_changes[~np.isnan(long_changes)]
    
    # Return max and min changes (or NaN if no valid changes)
    return pd.Series({
        'max_lat_change_last_5_steps': np.max(lat_changes) if len(lat_changes) > 0 else np.nan,
        'min_lat_change_last_5_steps': np.min(lat_changes) if len(lat_changes) > 0 else np.nan,
        'avg_lat_change_last_5_steps': np.mean(lat_changes) if len(lat_changes) > 0 else np.nan,
        'max_long_change_last_5_steps': np.max(long_changes) if len(long_changes) > 0 else np.nan,
        'min_long_change_last_5_steps': np.min(long_changes) if len(long_changes) > 0 else np.nan,
        'avg_long_change_last_5_steps': np.mean(long_changes) if len(long_changes) > 0 else np.nan,
    })

# Apply the calculation to each row
changes = train_df.apply(calculate_max_min_changes, axis=1)
train_df = pd.concat([train_df, changes], axis=1)
train_df = train_df.dropna()
train_df = train_df.reset_index(drop=True)

train_df.head(20)

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,time_position_4_steps_ago,latitude_5_steps_ago,longitude_5_steps_ago,time_position_5_steps_ago,max_lat_change_last_5_steps,min_lat_change_last_5_steps,avg_lat_change_last_5_steps,max_long_change_last_5_steps,min_long_change_last_5_steps,avg_long_change_last_5_steps
0,2024-01-12 15:54:48,307.6,16.1,5,313,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,...,2024-01-12 14:31:00,7.50361,77.58340,2024-01-12 14:07:47,-0.04094,-0.07741,-0.061848,0.10101,0.05438,0.080386
1,2024-01-12 16:14:59,309.5,16.1,-6,313,0,01-14 23:30,7.86929,77.11032,61e9f38eb937134a3c4bfd8b,...,2024-01-12 14:57:23,7.57302,77.49505,2024-01-12 14:31:00,-0.04094,-0.07741,-0.059254,0.10101,0.05438,0.076946
2,2024-01-12 16:35:24,308.7,16.0,2,311,0,01-14 23:30,7.92585,77.03811,61e9f38eb937134a3c4bfd8b,...,2024-01-12 15:18:48,7.65043,77.39404,2024-01-12 14:57:23,-0.04094,-0.06232,-0.055084,0.08010,0.05438,0.071186
3,2024-01-12 16:55:24,310.4,16.0,-1,311,0,01-14 23:30,7.98258,76.96880,61e9f38eb937134a3c4bfd8b,...,2024-01-12 15:39:47,7.71275,77.31394,2024-01-12 15:18:48,-0.04094,-0.05916,-0.053966,0.07809,0.05438,0.069028
4,2024-01-12 17:14:36,307.5,16.1,6,307,0,01-14 23:30,8.03598,76.90095,61e9f38eb937134a3c4bfd8b,...,2024-01-12 15:54:48,7.77191,77.23585,2024-01-12 15:39:47,-0.04094,-0.05673,-0.052814,0.07221,0.05438,0.066980
5,2024-01-12 17:36:36,322.2,16.2,-4,319,0,01-14 23:30,8.10476,76.83078,61e9f38eb937134a3c4bfd8b,...,2024-01-12 16:14:59,7.81285,77.18147,2024-01-12 15:54:48,-0.05340,-0.06878,-0.058382,0.07221,0.06785,0.070138
6,2024-01-12 17:57:36,294.2,15.9,3,292,0,01-14 23:30,8.15948,76.75688,61e9f38eb937134a3c4bfd8b,...,2024-01-12 16:35:24,7.86929,77.11032,2024-01-12 16:14:59,-0.05340,-0.06878,-0.058038,0.07390,0.06785,0.070688
7,2024-01-12 18:16:12,297.3,15.9,-5,294,0,01-14 23:30,8.19764,76.68344,61e9f38eb937134a3c4bfd8b,...,2024-01-12 16:55:24,7.92585,77.03811,2024-01-12 16:35:24,-0.03816,-0.06878,-0.054358,0.07390,0.06785,0.070934
8,2024-01-12 19:19:00,308.1,16.8,-5,303,0,01-14 23:30,8.37828,76.45826,61e9f38eb937134a3c4bfd8b,...,2024-01-12 17:14:36,7.98258,76.96880,2024-01-12 16:55:24,-0.03816,-0.18064,-0.079140,0.22518,0.06785,0.102108
9,2024-01-26 04:17:54,227.6,14.0,-15,226,0,01-14 23:30,-29.67649,31.39163,61e9f38eb937134a3c4bfd8b,...,2024-01-12 17:36:36,8.03598,76.90095,2024-01-12 17:14:36,38.05477,-0.18064,7.542494,45.06663,0.07017,9.101864


#### TEMPORAL FEATURES

In [3]:
import pandas as pd

# Reference times
reference_start_time = pd.to_datetime("2024-01-01 00:00:00")
reference_end_time = pd.to_datetime("2025-01-01 00:00:00")  # Might need to adjust the end time
total_time_span = (reference_end_time - reference_start_time).total_seconds()

# Training data - time feature conversion and normalization
train_df['original_time_converted'] = pd.to_datetime(train_df['time'], errors='coerce')

train_df['time_position_1_steps_ago_converted'] = pd.to_datetime(train_df['time_position_1_steps_ago'], errors='coerce')
train_df['time_position_2_steps_ago_converted'] = pd.to_datetime(train_df['time_position_2_steps_ago'], errors='coerce')
train_df['time_position_3_steps_ago_converted'] = pd.to_datetime(train_df['time_position_3_steps_ago'], errors='coerce')
train_df['time_position_4_steps_ago_converted'] = pd.to_datetime(train_df['time_position_4_steps_ago'], errors='coerce')
train_df['time_position_5_steps_ago_converted'] = pd.to_datetime(train_df['time_position_5_steps_ago'], errors='coerce')


train_df['time'] = (train_df['original_time_converted'] - reference_start_time).dt.total_seconds()

train_df['time_position_1_steps_ago'] = (train_df['time_position_1_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_2_steps_ago'] = (train_df['time_position_2_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_3_steps_ago'] = (train_df['time_position_3_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_4_steps_ago'] = (train_df['time_position_4_steps_ago_converted'] - reference_start_time).dt.total_seconds()
train_df['time_position_5_steps_ago'] = (train_df['time_position_5_steps_ago_converted'] - reference_start_time).dt.total_seconds()


# Normalize time (already between 0 and 1)
train_df['time'] = train_df['time'] / total_time_span
train_df['time_position_1_steps_ago'] = train_df['time_position_1_steps_ago'] / total_time_span
train_df['time_position_2_steps_ago'] = train_df['time_position_2_steps_ago'] / total_time_span
train_df['time_position_3_steps_ago'] = train_df['time_position_3_steps_ago'] / total_time_span
train_df['time_position_4_steps_ago'] = train_df['time_position_4_steps_ago'] / total_time_span
train_df['time_position_5_steps_ago'] = train_df['time_position_5_steps_ago'] / total_time_span


# Add new features
train_df['month_of_the_year'] = train_df['original_time_converted'].dt.month  # Month (1-12)
train_df['week_of_the_year'] = train_df['original_time_converted'].dt.isocalendar().week  # Week (1-53)
train_df['day_of_the_year'] = train_df['original_time_converted'].dt.dayofyear  # Day of year (1-365)
train_df['day_of_the_month'] = train_df['original_time_converted'].dt.day  # Day of month (1-31)
train_df['day_of_the_week'] = train_df['original_time_converted'].dt.dayofweek  # Day of week (0-6, where 0 is Monday)
train_df['hour_of_the_day'] = train_df['original_time_converted'].dt.hour  # Hour (0-23)
train_df['hours_passed'] = train_df['original_time_converted'].diff().dt.total_seconds() / 3600
hours = train_df['original_time_converted'].dt.hour
minutes = train_df['original_time_converted'].dt.minute

# Convert hour to cyclical features
train_df['hour_sin'] = (np.sin(2 * np.pi * hours / 24) + 1) / 2
train_df['hour_cos'] = (np.cos(2 * np.pi * hours / 24) + 1) / 2

# Convert minute to cyclical features and normalize to [0,1]
train_df['minute_sin'] = (np.sin(2 * np.pi * minutes / 60) + 1) / 2
train_df['minute_cos'] = (np.cos(2 * np.pi * minutes / 60) + 1) / 2

# Normalize other features
# Min-max normalization to scale features between 0 and 1
train_df['month_of_the_year'] = (train_df['month_of_the_year'] - 1) / 11  # Normalize month (1-12)
train_df['week_of_the_year'] = (train_df['week_of_the_year'] - 1) / 52  # Normalize week (1-53)
train_df['day_of_the_year'] = (train_df['day_of_the_year'] - 1) / 365  # Normalize day of year (1-365)
train_df['day_of_the_month'] = (train_df['day_of_the_month'] - 1) / 30  # Normalize day of month (1-31)
train_df['day_of_the_week'] = train_df['day_of_the_week'] / 6  # Normalize day of week (0-6)
train_df['hour_of_the_day'] = train_df['hour_of_the_day'] / 23  # Normalize hour (0-23)
train_df['hours_passed'] = train_df['hours_passed'] / 24

# Add time diff features
train_df = train_df.sort_values(['vesselId', 'original_time_converted'])
train_df['time_diff'] = train_df.groupby('vesselId')['original_time_converted'].diff(-1)  # Using -1 for difference with next row

# Create boolean features for different time thresholds
train_df['time_diff_gt_10min'] = ((-train_df['time_diff'].dt.total_seconds()) > 10 * 60).astype(int)
train_df['time_diff_gt_20min'] = ((-train_df['time_diff'].dt.total_seconds()) > 20 * 60).astype(int)
train_df['time_diff_gt_40min'] = ((-train_df['time_diff'].dt.total_seconds()) > 40 * 60).astype(int)
train_df['time_diff_gt_1hour'] = ((-train_df['time_diff'].dt.total_seconds()) > 1 * 3600).astype(int)
train_df['time_diff_gt_2hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 2 * 3600).astype(int)
train_df['time_diff_gt_6hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 6 * 3600).astype(int)
train_df['time_diff_gt_12hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 12 * 3600).astype(int)
train_df['time_diff_gt_1day'] = ((-train_df['time_diff'].dt.total_seconds()) > 24 * 3600).astype(int)

# Fill NA values (last row of each vessel group) with 0
time_diff_columns = [col for col in train_df.columns if col.startswith('time_diff_gt_')]
train_df[time_diff_columns] = train_df[time_diff_columns].fillna(0)

# Drop the temporary time_diff column
train_df.drop('time_diff', axis=1, inplace=True)


# Drop intermediate columns
train_df.drop(['original_time_converted',
               'time_position_1_steps_ago_converted',
               'time_position_2_steps_ago_converted',
               'time_position_3_steps_ago_converted',
               'time_position_4_steps_ago_converted',
               'time_position_5_steps_ago_converted',
               ], axis=1, inplace=True)

# Test data - time feature conversion and normalization
test_df['time_converted'] = pd.to_datetime(test_df['time'], errors='coerce')
test_df['time'] = (test_df['time_converted'] - reference_start_time).dt.total_seconds()
test_df['time'] = test_df['time'] / total_time_span

# Add new features
test_df['month_of_the_year'] = test_df['time_converted'].dt.month  # Month (1-12)
test_df['week_of_the_year'] = test_df['time_converted'].dt.isocalendar().week  # Week (1-53)
test_df['day_of_the_year'] = test_df['time_converted'].dt.dayofyear  # Day of year (1-365)
test_df['day_of_the_month'] = test_df['time_converted'].dt.day  # Day of month (1-31)
test_df['day_of_the_week'] = test_df['time_converted'].dt.dayofweek  # Day of week (0-6, where 0 is Monday)
test_df['hour_of_the_day'] = test_df['time_converted'].dt.hour  # Hour (0-23)

test_df['hours_passed'] = test_df['time_converted'].diff().dt.total_seconds() / 3600
hours = test_df['time_converted'].dt.hour
minutes = test_df['time_converted'].dt.minute

# Convert hour to cyclical features
test_df['hour_sin'] = (np.sin(2 * np.pi * hours / 24) + 1) / 2
test_df['hour_cos'] = (np.cos(2 * np.pi * hours / 24) + 1) / 2

# Convert minute to cyclical features and normalize to [0,1]
test_df['minute_sin'] = (np.sin(2 * np.pi * minutes / 60) + 1) / 2
test_df['minute_cos'] = (np.cos(2 * np.pi * minutes / 60) + 1) / 2


# Normalize other features in test data
test_df['month_of_the_year'] = (test_df['month_of_the_year'] - 1) / 11  # Normalize month (1-12)
test_df['week_of_the_year'] = (test_df['week_of_the_year'] - 1) / 52  # Normalize week (1-53)
test_df['day_of_the_year'] = (test_df['day_of_the_year'] - 1) / 365  # Normalize day of year (1-365)
test_df['day_of_the_month'] = (test_df['day_of_the_month'] - 1) / 30  # Normalize day of month (1-31)
test_df['day_of_the_week'] = test_df['day_of_the_week'] / 6  # Normalize day of week (0-6)
test_df['hour_of_the_day'] = test_df['hour_of_the_day'] / 23  # Normalize hour (0-23)

# Add time diff features
test_df = test_df.sort_values(['vesselId', 'time_converted'])
test_df['time_diff'] = test_df.groupby('vesselId')['time_converted'].diff(-1)  # Using -1 for difference with next row


# Create boolean features for different time thresholds
test_df['time_diff_gt_10min'] = ((-test_df['time_diff'].dt.total_seconds()) > 10 * 60).astype(int)
test_df['time_diff_gt_20min'] = ((-test_df['time_diff'].dt.total_seconds()) > 20 * 60).astype(int)
test_df['time_diff_gt_40min'] = ((-test_df['time_diff'].dt.total_seconds()) > 40 * 60).astype(int)
test_df['time_diff_gt_1hour'] = ((-test_df['time_diff'].dt.total_seconds()) > 1 * 3600).astype(int)
test_df['time_diff_gt_2hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 2 * 3600).astype(int)
test_df['time_diff_gt_6hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 6 * 3600).astype(int)
test_df['time_diff_gt_12hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 12 * 3600).astype(int)
test_df['time_diff_gt_1day'] = ((-test_df['time_diff'].dt.total_seconds()) > 24 * 3600).astype(int)

# Fill NA values (last row of each vessel group) with 0
time_diff_columns = [col for col in test_df.columns if col.startswith('time_diff_gt_')]
test_df[time_diff_columns] = test_df[time_diff_columns].fillna(0)

# Drop the temporary time_diff column
test_df.drop('time_diff', axis=1, inplace=True)

# Drop intermediate columns
test_df.drop(['time_converted'], axis=1, inplace=True)


In [4]:
train_df.tail()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
1517014,0.349568,324.1,13.5,-2,325,0,05-08 03:00,59.63337,21.43237,clh6aqawa0007gh0z9h6zi9bo,...,0.206107,0.095492,1,1,0,0,0,0,0,0
1517015,0.349607,324.2,13.3,-3,326,0,05-08 03:00,59.69588,21.34225,clh6aqawa0007gh0z9h6zi9bo,...,0.345492,0.975528,1,1,0,0,0,0,0,0
1517016,0.349647,356.5,12.2,-1,354,0,05-08 03:00,59.76388,21.35317,clh6aqawa0007gh0z9h6zi9bo,...,0.989074,0.396044,1,1,0,0,0,0,0,0
1517017,0.349685,52.6,17.3,3,50,0,05-08 03:00,59.83316,21.38489,clh6aqawa0007gh0z9h6zi9bo,...,0.128428,0.165435,1,1,0,0,0,0,0,0
1517018,0.349725,53.6,17.7,-1,51,0,05-08 03:00,59.89167,21.54685,clh6aqawa0007gh0z9h6zi9bo,...,0.447736,0.997261,0,0,0,0,0,0,0,0


In [5]:
test_df.tail()

,ID,vesselId,time,scaling_factor,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,day_of_the_week,hour_of_the_day,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
51161,51161,clh6aqawa0007gh0z9h6zi9bo,0.363232,0.1,0.363636,0.346154,0.361644,0.366667,1.0,0.956522,...,0.165435,0.128428,1,1,0,0,0,0,0,0
51302,51302,clh6aqawa0007gh0z9h6zi9bo,0.363270,0.1,0.363636,0.346154,0.361644,0.366667,1.0,0.956522,...,0.396044,0.989074,1,1,0,0,0,0,0,0
51444,51444,clh6aqawa0007gh0z9h6zi9bo,0.363309,0.1,0.363636,0.346154,0.361644,0.366667,1.0,1.000000,...,0.975528,0.345492,1,1,0,0,0,0,0,0
51595,51595,clh6aqawa0007gh0z9h6zi9bo,0.363349,0.1,0.363636,0.346154,0.361644,0.366667,1.0,1.000000,...,0.095492,0.206107,1,0,0,0,0,0,0,0
51654,51654,clh6aqawa0007gh0z9h6zi9bo,0.363386,0.1,0.363636,0.346154,0.361644,0.366667,1.0,1.000000,...,0.447736,0.997261,0,0,0,0,0,0,0,0


#### LINEAR REGRESSION PREDICTION BASED ON LAST POSITIONS

Tried Non-linear, but more errors

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm.auto import tqdm

def linreg_predict_position(row):
    """
    Fits lines to historical latitude and longitude data and predicts next positions.
    
    Args:
        row: A pandas Series containing historical position data
        
    Returns:
        tuple: (predicted_latitude, predicted_longitude)
    """
    # Extract historical latitudes and longitudes
    lats = np.array([
        row['latitude_5_steps_ago'],
        row['latitude_4_steps_ago'],
        row['latitude_3_steps_ago'],
        row['latitude_2_steps_ago'],
        row['latitude_1_steps_ago']
    ])
    
    longs = np.array([
        row['longitude_5_steps_ago'],
        row['longitude_4_steps_ago'],
        row['longitude_3_steps_ago'],
        row['longitude_2_steps_ago'],
        row['longitude_1_steps_ago']
    ])
    
    # Extract corresponding times
    times = np.array([
        row['time_position_5_steps_ago'],
        row['time_position_4_steps_ago'],
        row['time_position_3_steps_ago'],
        row['time_position_2_steps_ago'],
        row['time_position_1_steps_ago']
    ])
    
    # Reshape for sklearn
    times = times.reshape(-1, 1)
    
    # Fit latitude line
    lat_model = LinearRegression()
    lat_model.fit(times, lats)
    
    # Fit longitude line
    long_model = LinearRegression()
    long_model.fit(times, longs)
    
    # Predict for current time
    current_time = np.array([[row['time']]])
    predicted_lat = lat_model.predict(current_time)[0]
    predicted_long = long_model.predict(current_time)[0]
    
    return predicted_lat, predicted_long

def add_predictions(df):
    """
    Adds predicted positions to the dataframe with a progress bar
    """
    # Create empty lists to store predictions
    lat_predictions = []
    long_predictions = []
    
    # Iterate through dataframe with progress bar
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Calculating predictions"):
        pred_lat, pred_long = linreg_predict_position(row)
        lat_predictions.append(pred_lat)
        long_predictions.append(pred_long)
    
    # Add predictions to dataframe
    df['linreg_predicted_latitude'] = lat_predictions
    df['linreg_predicted_longitude'] = long_predictions
    
    # Add error calculations
    df['linreg_latitude_error'] = np.abs(df['latitude'] - df['linreg_predicted_latitude'])
    df['linreg_longitude_error'] = np.abs(df['longitude'] - df['linreg_predicted_longitude'])
    
    return df

train_df = add_predictions(train_df)

train_df.head()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Calculating predictions: 100%|██████████| 1517019/1517019 [09:44<00:00, 2593.66it/s]


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day,linreg_predicted_latitude,linreg_predicted_longitude,linreg_latitude_error,linreg_longitude_error
0,0.031866,307.6,16.1,5,313,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.817029,77.178165,0.004179,0.003305
1,0.031905,309.5,16.1,-6,313,0,01-14 23:30,7.86929,77.11032,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.872235,77.104615,0.002945,0.005705
2,0.031943,308.7,16.0,2,311,0,01-14 23:30,7.92585,77.03811,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.927388,77.033801,0.001538,0.004309
3,0.031981,310.4,16.0,-1,311,0,01-14 23:30,7.98258,76.96880,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.981538,76.965162,0.001042,0.003638
4,0.032018,307.5,16.1,6,307,0,01-14 23:30,8.03598,76.90095,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,8.035558,76.900286,0.000422,0.000664


In [7]:
train_df.head(50)

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day,linreg_predicted_latitude,linreg_predicted_longitude,linreg_latitude_error,linreg_longitude_error
0,0.031866,307.6,16.1,5,313,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.817029,77.178165,0.004179,0.003305
1,0.031905,309.5,16.1,-6,313,0,01-14 23:30,7.86929,77.11032,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.872235,77.104615,0.002945,0.005705
2,0.031943,308.7,16.0,2,311,0,01-14 23:30,7.92585,77.03811,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.927388,77.033801,0.001538,0.004309
3,0.031981,310.4,16.0,-1,311,0,01-14 23:30,7.98258,76.96880,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.981538,76.965162,0.001042,0.003638
4,0.032018,307.5,16.1,6,307,0,01-14 23:30,8.03598,76.90095,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,8.035558,76.900286,0.000422,0.000664
5,0.032059,322.2,16.2,-4,319,0,01-14 23:30,8.10476,76.83078,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,8.097533,76.823687,0.007227,0.007093
6,0.032099,294.2,15.9,3,292,0,01-14 23:30,8.15948,76.75688,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,8.162317,76.755926,0.002837,0.000954
7,0.032135,297.3,15.9,-5,294,0,01-14 23:30,8.19764,76.68344,61e9f38eb937134a3c4bfd8b,...,1,1,0,0,0,0,8.214580,76.693790,0.016940,0.010350
8,0.032254,308.1,16.8,-5,303,0,01-14 23:30,8.37828,76.45826,61e9f38eb937134a3c4bfd8b,...,1,1,1,1,1,1,8.374970,76.468905,0.003310,0.010645
9,0.068795,227.6,14.0,-15,226,0,01-14 23:30,-29.67649,31.39163,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,60.601036,7.338810,90.277526,24.052820


In [8]:
train_df.head(50)

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day,linreg_predicted_latitude,linreg_predicted_longitude,linreg_latitude_error,linreg_longitude_error
0,0.031866,307.6,16.1,5,313,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.817029,77.178165,0.004179,0.003305
1,0.031905,309.5,16.1,-6,313,0,01-14 23:30,7.86929,77.11032,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.872235,77.104615,0.002945,0.005705
2,0.031943,308.7,16.0,2,311,0,01-14 23:30,7.92585,77.03811,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.927388,77.033801,0.001538,0.004309
3,0.031981,310.4,16.0,-1,311,0,01-14 23:30,7.98258,76.96880,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,7.981538,76.965162,0.001042,0.003638
4,0.032018,307.5,16.1,6,307,0,01-14 23:30,8.03598,76.90095,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,8.035558,76.900286,0.000422,0.000664
5,0.032059,322.2,16.2,-4,319,0,01-14 23:30,8.10476,76.83078,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,8.097533,76.823687,0.007227,0.007093
6,0.032099,294.2,15.9,3,292,0,01-14 23:30,8.15948,76.75688,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,8.162317,76.755926,0.002837,0.000954
7,0.032135,297.3,15.9,-5,294,0,01-14 23:30,8.19764,76.68344,61e9f38eb937134a3c4bfd8b,...,1,1,0,0,0,0,8.214580,76.693790,0.016940,0.010350
8,0.032254,308.1,16.8,-5,303,0,01-14 23:30,8.37828,76.45826,61e9f38eb937134a3c4bfd8b,...,1,1,1,1,1,1,8.374970,76.468905,0.003310,0.010645
9,0.068795,227.6,14.0,-15,226,0,01-14 23:30,-29.67649,31.39163,61e9f38eb937134a3c4bfd8b,...,0,0,0,0,0,0,60.601036,7.338810,90.277526,24.052820


#### PORT LAT AND LONG

In [9]:
import pandas as pd
import numpy as np

# ADDS PORT LAT AND LONG TO TEST SET BASED ON SCHEDULES

def engineer_port_locations(schedule_df, test_df):
    """
    Engineer port latitude and longitude features for test data based on vessel schedules.
    Handles timezone-aware datetime comparisons.
    
    Parameters:
    schedule_df (pd.DataFrame): Schedule data with vessel ports and times
    test_df (pd.DataFrame): Test data requiring port location features
    
    Returns:
    pd.DataFrame: Test data with added port_lat and port_long features
    """
    # Convert string data to datetime and ensure timezone consistency
    schedule_df = schedule_df.copy()
    test_df = test_df.copy()
    
    # Convert schedule dates to UTC
    schedule_df['arrivalDate'] = pd.to_datetime(schedule_df['arrivalDate']).dt.tz_convert('UTC')
    schedule_df['sailingDate'] = pd.to_datetime(schedule_df['sailingDate']).dt.tz_convert('UTC')
    
    # Convert test dates to UTC
    test_df['time_utc'] = pd.to_datetime(test_df['time']).dt.tz_localize('UTC')
    
    # Initialize new columns
    test_df['port_lat'] = np.nan
    test_df['port_long'] = np.nan
    
    # Process each row in test_df
    for idx, row in test_df.iterrows():
        vessel_schedule = schedule_df[schedule_df['vesselId'] == row['vesselId']].copy()
        
        if len(vessel_schedule) == 0:
            continue
            
        # Sort vessel schedule by arrival date
        vessel_schedule = vessel_schedule.sort_values('arrivalDate')
        
        # Find the relevant port
        # Case 1: Vessel is at a port (between arrival and sailing)
        at_port = vessel_schedule[
            (vessel_schedule['arrivalDate'] <= row['time_utc']) & 
            (vessel_schedule['sailingDate'] >= row['time_utc'])
        ]
        
        if len(at_port) > 0:
            # Use the current port's coordinates
            test_df.at[idx, 'port_lat'] = at_port.iloc[0]['portLatitude']
            test_df.at[idx, 'port_long'] = at_port.iloc[0]['portLongitude']
            continue
        
        # Case 2: Vessel is between ports
        next_port = vessel_schedule[vessel_schedule['arrivalDate'] > row['time_utc']].iloc[0] if len(vessel_schedule[vessel_schedule['arrivalDate'] > row['time_utc']]) > 0 else None
        prev_port = vessel_schedule[vessel_schedule['sailingDate'] < row['time_utc']].iloc[-1] if len(vessel_schedule[vessel_schedule['sailingDate'] < row['time_utc']]) > 0 else None
        
        if prev_port is not None and next_port is not None:
            # Calculate time ratios
            total_time = (next_port['arrivalDate'] - prev_port['sailingDate']).total_seconds()
            elapsed_time = (row['time_utc'] - prev_port['sailingDate']).total_seconds()
            ratio = elapsed_time / total_time
            
            # Interpolate coordinates
            test_df.at[idx, 'port_lat'] = prev_port['portLatitude'] + (next_port['portLatitude'] - prev_port['portLatitude']) * ratio
            test_df.at[idx, 'port_long'] = prev_port['portLongitude'] + (next_port['portLongitude'] - prev_port['portLongitude']) * ratio
        
        elif prev_port is not None:
            # Use last known port
            test_df.at[idx, 'port_lat'] = prev_port['portLatitude']
            test_df.at[idx, 'port_long'] = prev_port['portLongitude']
        
        elif next_port is not None:
            # Use next port
            test_df.at[idx, 'port_lat'] = next_port['portLatitude']
            test_df.at[idx, 'port_long'] = next_port['portLongitude']

    test_df.drop(["time_utc"], axis=1, inplace=True)
    
    return test_df

test_df = engineer_port_locations(test_df=test_df, schedule_df=schedules_df)



In [10]:
# ADDS PORT LAT AND LONG TO TRAIN USING PORT ID MAPPING

def add_port_coordinates(df, ports_df):
    """
    Add port latitude and longitude columns to the input dataframe by mapping from ports_df.
    
    Parameters:
    df (pandas.DataFrame): Input dataframe containing portId column
    ports_df (pandas.DataFrame): Ports reference dataframe containing portId, latitude, and longitude
    
    Returns:
    pandas.DataFrame: Original dataframe with two new columns: port_lat and port_long
    """
    # Create a mapping dictionary for faster lookup
    port_coords = ports_df.set_index('portId')[['latitude', 'longitude']].to_dict('index')
    
    # Initialize new columns
    df['port_lat'] = float('nan')
    df['port_long'] = float('nan')
    
    # Update values for rows with valid portId
    for port_id, coords in port_coords.items():
        mask = df['portId'] == port_id
        df.loc[mask, 'port_lat'] = coords['latitude']
        df.loc[mask, 'port_long'] = coords['longitude']
    
    return df

# Apply the function to train_df
train_df = add_port_coordinates(train_df, ports_df)

# Verify the results
print("\nSample of updated training data:")
print(train_df[['portId', 'port_lat', 'port_long']].head())

# Check for any rows where mapping failed
unmapped = train_df[train_df['port_lat'].isna()]['portId'].nunique()
print(f"\nNumber of unique portIds with no mapping: {unmapped}")




Sample of updated training data:
                     portId   port_lat  port_long
0  61d376d893c6feb83e5eb546  18.941944  72.885278
1  61d376d893c6feb83e5eb546  18.941944  72.885278
2  61d376d893c6feb83e5eb546  18.941944  72.885278
3  61d376d893c6feb83e5eb546  18.941944  72.885278
4  61d376d893c6feb83e5eb546  18.941944  72.885278

Number of unique portIds with no mapping: 0


In [11]:
# PREPARING FOR PORT PREDICTION

port_predictor_features = ['time',
       'month_of_the_year', 'week_of_the_year', 'day_of_the_year',
       'day_of_the_month', 'day_of_the_week', 'hour_of_the_day',
       'hours_passed', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
       'time_diff_gt_10min', 'time_diff_gt_20min', 'time_diff_gt_40min',
       'time_diff_gt_1hour', 'time_diff_gt_2hours', 'time_diff_gt_6hours',
       'time_diff_gt_12hours', 'time_diff_gt_1day']

vessel_ids_lacking_port = test_df[
    (test_df['port_lat'].isna()) | 
    (test_df['port_long'].isna())
]['vesselId'].unique()

print(f"Number of vessels with missing port coordinates: {len(vessel_ids_lacking_port)}")

Number of vessels with missing port coordinates: 157


In [12]:
# PORT ID CLASSIFICATION - TRAIN CLASSIFIERS

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

def filter_vessel_data(vessel_data, min_samples_per_port=2):
    """
    Filter vessel data to ensure minimum samples per port
    """
    port_counts = vessel_data['portId'].value_counts()
    valid_ports = port_counts[port_counts >= min_samples_per_port].index
    return vessel_data[vessel_data['portId'].isin(valid_ports)]

def train_port_classifier(vessel_id, train_df, features, min_samples_per_port=2):
    """
    Train a port classifier for a specific vessel
    """
    # Filter data for this vessel
    vessel_data = train_df[train_df['vesselId'] == vessel_id].copy()
    
    # Skip if no data or no valid port IDs
    if len(vessel_data) == 0 or vessel_data['portId'].isna().all():
        return None, None, {'error': 'No valid data'}
    
    # Remove rows with NaN port IDs
    vessel_data = vessel_data.dropna(subset=['portId'])
    
    # Filter to ensure minimum samples per port
    vessel_data = filter_vessel_data(vessel_data, min_samples_per_port)
    
    # Get unique ports for this vessel
    unique_ports = vessel_data['portId'].unique()
    
    # If fewer than 2 ports remain after filtering, skip training
    if len(unique_ports) < 2:
        return None, unique_ports, {'error': f'Unique port: {unique_ports}'}
    
    # Prepare features and target
    X = vessel_data[features]
    y = vessel_data['portId']
    
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Initialize model
    classifier = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    )
    
    # Train model
    classifier.fit(X_train, y_train)
    
    # Make predictions on validation set
    y_pred = classifier.predict(X_val)
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_val, y_pred),
        'training_samples': len(X_train),
        'unique_ports': len(unique_ports),
    }
    
    return classifier, unique_ports, metrics
        

# Dictionary to store models for each vessel
vessel_classifiers = {}

# Train classifiers for each vessel with sufficient data
MIN_SAMPLES_PER_PORT = 2

for vessel_id in vessel_ids_lacking_port:
    classifier, unique_ports, metrics = train_port_classifier(
        vessel_id, train_df, port_predictor_features, 
        min_samples_per_port=MIN_SAMPLES_PER_PORT
    )
    
    if metrics is not None and 'error' not in metrics:
        vessel_classifiers[vessel_id] = {
            'classifier': classifier,
            #'unique_ports': unique_ports,
            'metrics': metrics
        }

# Print summary
print(f"\nSuccessfully trained classifiers for {len(vessel_classifiers)} vessels")
print(f"Failed to train classifiers for {len(vessel_ids_lacking_port) - len(vessel_classifiers)} vessels")

accuracies = [model['metrics']['accuracy'] for model in vessel_classifiers.values()]

# Calculate statistics
mean_acc = np.mean(accuracies)
median_acc = np.median(accuracies)
min_acc = np.min(accuracies)
max_acc = np.max(accuracies)

print(f"Classifier Performance Statistics:")
print(f"Mean accuracy: {mean_acc:.3f}")
print(f"Median accuracy: {median_acc:.3f}")
print(f"Min accuracy: {min_acc:.3f}")
print(f"Max accuracy: {max_acc:.3f}")


Successfully trained classifiers for 155 vessels
Failed to train classifiers for 2 vessels
Classifier Performance Statistics:
Mean accuracy: 0.947
Median accuracy: 0.957
Min accuracy: 0.612
Max accuracy: 1.000


In [13]:
# CLASSIFY PORT ID FOR TEST SET AND ADD CORRESPONDING LAT AND LONG

def fill_missing_port_coordinates(test_df, vessel_classifiers, ports_df):
    """
    Fill missing port coordinates in test_df using trained classifiers or unique ports
    """
    # Create a copy of test_df to avoid modifying the original
    df = test_df.copy()
    
    # Create a mapping of portId to coordinates
    port_coords = ports_df.set_index('portId')[['latitude', 'longitude']].to_dict('index')
    
    # Identify rows with missing coordinates
    missing_coords_mask = df['port_lat'].isna() | df['port_long'].isna()
    
    # Process each vessel's data
    for vessel_id in df[missing_coords_mask]['vesselId'].unique():
        # Get rows for this vessel that need predictions
        vessel_mask = (df['vesselId'] == vessel_id) & missing_coords_mask
        vessel_rows = df[vessel_mask]
        
        if len(vessel_rows) == 0:
            continue
            
        # Check if we have a classifier for this vessel
        vessel_data = train_port_classifier(vessel_id, train_df, port_predictor_features)
        
        if vessel_data[0] is None:  # No classifier, use unique port
            if vessel_data[1] is not None and len(vessel_data[1]) > 0:
                predicted_port = vessel_data[1][0]  # Take the first (only) unique port
                
                # Fill coordinates for this port
                if predicted_port in port_coords:
                    df.loc[vessel_mask, 'port_lat'] = port_coords[predicted_port]['latitude']
                    df.loc[vessel_mask, 'port_long'] = port_coords[predicted_port]['longitude']
        
        else:  # Use classifier
            classifier = vessel_classifiers[vessel_id]['classifier']
            
            # Get features for prediction
            X_pred = vessel_rows[port_predictor_features]
            
            # Make predictions
            predicted_ports = classifier.predict(X_pred)
            
            # Fill coordinates for each prediction
            for i, (idx, row) in enumerate(vessel_rows.iterrows()):
                predicted_port = predicted_ports[i]
                if predicted_port in port_coords:
                    df.loc[idx, 'port_lat'] = port_coords[predicted_port]['latitude']
                    df.loc[idx, 'port_long'] = port_coords[predicted_port]['longitude']
    
    return df

# Apply the function
test_df_filled = fill_missing_port_coordinates(test_df, vessel_classifiers, ports_df)

# Print summary of changes
original_nulls = test_df['port_lat'].isna().sum() + test_df['port_long'].isna().sum()
remaining_nulls = test_df_filled['port_lat'].isna().sum() + test_df_filled['port_long'].isna().sum()

print(f"\nOriginal missing coordinates: {original_nulls}")
print(f"Remaining missing coordinates: {remaining_nulls}")
print(f"Filled {original_nulls - remaining_nulls} coordinates")

test_df = test_df_filled

test_df.head()


Original missing coordinates: 75786
Remaining missing coordinates: 0
Filled 75786 coordinates


,ID,vesselId,time,scaling_factor,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,day_of_the_week,hour_of_the_day,...,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day,port_lat,port_long
4,4,61e9f38eb937134a3c4bfd8d,0.349750,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,1,1,0,0,0,0,0,0,48.380556,-4.474167
201,201,61e9f38eb937134a3c4bfd8d,0.349802,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,1,1,1,0,0,0,0,0,48.380556,-4.474167
583,583,61e9f38eb937134a3c4bfd8d,0.349904,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,1,0,0,0,0,0,0,0,48.380556,-4.474167
701,701,61e9f38eb937134a3c4bfd8d,0.349938,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,1,0,0,0,0,0,0,0,48.380556,-4.474167
829,829,61e9f38eb937134a3c4bfd8d,0.349961,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.086957,...,1,1,0,0,0,0,0,0,48.380556,-4.474167


#### PORT BASED LAT LONG PREDICTIONS

In [14]:
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm.auto import tqdm

def predict_position_with_port(row):
    """
    Predicts next position considering both recent movement and destination port.
    Uses a weighted combination of:
    1. The direction to port
    2. The recent movement trend
    
    Args:
        row: A pandas Series containing position data and port information
        
    Returns:
        tuple: (predicted_latitude, predicted_longitude)
    """
    # Current position (actually last known position)
    current_lat = row['latitude_1_steps_ago']
    current_long = row['longitude_1_steps_ago']
    
    # Calculate direction vector to port
    to_port_lat = row['port_lat'] - current_lat
    to_port_long = row['port_long'] - current_long
    
    # Normalize the port direction vector
    port_distance = np.sqrt(to_port_lat**2 + to_port_long**2)
    if port_distance > 0:
        to_port_lat = to_port_lat / port_distance
        to_port_long = to_port_long / port_distance
    
    # Get recent movement vector from averages
    recent_lat_change = row['avg_lat_change_last_5_steps']
    recent_long_change = row['avg_long_change_last_5_steps']
    
    # Normalize the recent movement vector
    recent_magnitude = np.sqrt(recent_lat_change**2 + recent_long_change**2)
    if recent_magnitude > 0:
        recent_lat_change = recent_lat_change / recent_magnitude
        recent_long_change = recent_long_change / recent_magnitude
    
    # Calculate weight based on distance to port
    # When far from port, rely more on recent movement
    # When closer to port, give more weight to port direction
    max_weight_distance = 1.0  # You might want to adjust this based on your data
    port_weight = max(0, min(1, port_distance / max_weight_distance))
    recent_weight = 1 - port_weight
    
    # Combine the two vectors with weights
    predicted_direction_lat = (recent_weight * recent_lat_change + 
                             port_weight * to_port_lat)
    predicted_direction_long = (recent_weight * recent_long_change + 
                              port_weight * to_port_long)
    
    # Normalize the final direction
    final_magnitude = np.sqrt(predicted_direction_lat**2 + predicted_direction_long**2)
    if final_magnitude > 0:
        predicted_direction_lat = predicted_direction_lat / final_magnitude
        predicted_direction_long = predicted_direction_long / final_magnitude
    
    # Scale by the recent average movement magnitude to get actual distance
    step_magnitude = np.sqrt(row['avg_lat_change_last_5_steps']**2 + 
                           row['avg_long_change_last_5_steps']**2)
    
    # Calculate final predictions
    predicted_lat = current_lat + predicted_direction_lat * step_magnitude
    predicted_long = current_long + predicted_direction_long * step_magnitude
    
    return predicted_lat, predicted_long

def add_port_based_predictions(df):
    """
    Adds port-based position predictions to the dataframe with a progress bar
    """
    lat_predictions = []
    long_predictions = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Calculating port-based predictions"):
        pred_lat, pred_long = predict_position_with_port(row)
        lat_predictions.append(pred_lat)
        long_predictions.append(pred_long)
    
    df['port_based_predicted_latitude'] = lat_predictions
    df['port_based_predicted_longitude'] = long_predictions
    
    # Add error calculations
    df['port_based_latitude_error'] = np.abs(df['latitude'] - df['port_based_predicted_latitude'])
    df['port_based_longitude_error'] = np.abs(df['longitude'] - df['port_based_predicted_longitude'])
    
    return df

# Add the port-based predictions
train_df = add_port_based_predictions(train_df)

Calculating port-based predictions: 100%|██████████| 1517019/1517019 [00:32<00:00, 47095.51it/s]


In [15]:
def add_avg_predictions(df):
    """
    Adds new features that average the linear regression and port-based predictions
    
    Args:
        df: DataFrame containing both linreg and port-based predictions
        
    Returns:
        DataFrame with new combined prediction features
    """
    # Calculate averages for latitude and longitude
    df['avg_linreg_port_predicted_latitude'] = (
        df['linreg_predicted_latitude'] + 
        df['port_based_predicted_latitude']
    ) / 2
    
    df['avg_linreg_port_predicted_longitude'] = (
        df['linreg_predicted_longitude'] + 
        df['port_based_predicted_longitude']
    ) / 2
    
    # Add error calculations for the combined predictions
    df['avg_linreg_port_latitude_error'] = np.abs(
        df['latitude'] - df['avg_linreg_port_predicted_latitude']
    )
    df['avg_linreg_port_longitude_error'] = np.abs(
        df['longitude'] - df['avg_linreg_port_predicted_longitude']
    )
    
    return df

train_df = add_avg_predictions(train_df)

train_df.head()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,port_lat,port_long,port_based_predicted_latitude,port_based_predicted_longitude,port_based_latitude_error,port_based_longitude_error,avg_linreg_port_predicted_latitude,avg_linreg_port_predicted_longitude,avg_linreg_port_latitude_error,avg_linreg_port_longitude_error
0,0.031866,307.6,16.1,5,313,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,7.866420,77.199040,0.053570,0.017570,7.841725,77.188602,0.028875,0.007132
1,0.031905,309.5,16.1,-6,313,0,01-14 23:30,7.86929,77.11032,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,7.903451,77.146495,0.034161,0.036175,7.887843,77.125555,0.018553,0.015235
2,0.031943,308.7,16.0,2,311,0,01-14 23:30,7.92585,77.03811,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,7.953385,77.078231,0.027535,0.040121,7.940387,77.056016,0.014537,0.017906
3,0.031981,310.4,16.0,-1,311,0,01-14 23:30,7.98258,76.96880,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,8.007837,77.007203,0.025257,0.038403,7.994688,76.986182,0.012108,0.017382
4,0.032018,307.5,16.1,6,307,0,01-14 23:30,8.03598,76.90095,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,8.062509,76.939018,0.026529,0.038068,8.049034,76.919652,0.013054,0.018702


In [16]:
print("Linear regression average errors:")
print(f"Latitude: {train_df['linreg_latitude_error'].mean():.6f}")
print(f"Longitude: {train_df['linreg_longitude_error'].mean():.6f}")

print("\nPort-based average errors:")
print(f"Latitude: {train_df['port_based_latitude_error'].mean():.6f}")
print(f"Longitude: {train_df['port_based_longitude_error'].mean():.6f}")

print("\LinReg/Port-based mean average errors:")
print(f"Latitude: {train_df['avg_linreg_port_predicted_latitude'].mean():.6f}")
print(f"Longitude: {train_df['avg_linreg_port_predicted_longitude'].mean():.6f}")

Linear regression average errors:
Latitude: 0.112335
Longitude: 0.237953

Port-based average errors:
Latitude: 0.189816
Longitude: 0.328309
\LinReg/Port-based mean average errors:
Latitude: 36.612447
Longitude: 11.509145


In [17]:

print(train_df.count())
train_df.head()

time                                   1517019
cog                                    1517019
sog                                    1517019
rot                                    1517019
heading                                1517019
                                        ...   
port_based_longitude_error             1517019
avg_linreg_port_predicted_latitude     1517019
avg_linreg_port_predicted_longitude    1517019
avg_linreg_port_latitude_error         1517019
avg_linreg_port_longitude_error        1517019
Length: 65, dtype: int64


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,port_lat,port_long,port_based_predicted_latitude,port_based_predicted_longitude,port_based_latitude_error,port_based_longitude_error,avg_linreg_port_predicted_latitude,avg_linreg_port_predicted_longitude,avg_linreg_port_latitude_error,avg_linreg_port_longitude_error
0,0.031866,307.6,16.1,5,313,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,7.866420,77.199040,0.053570,0.017570,7.841725,77.188602,0.028875,0.007132
1,0.031905,309.5,16.1,-6,313,0,01-14 23:30,7.86929,77.11032,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,7.903451,77.146495,0.034161,0.036175,7.887843,77.125555,0.018553,0.015235
2,0.031943,308.7,16.0,2,311,0,01-14 23:30,7.92585,77.03811,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,7.953385,77.078231,0.027535,0.040121,7.940387,77.056016,0.014537,0.017906
3,0.031981,310.4,16.0,-1,311,0,01-14 23:30,7.98258,76.96880,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,8.007837,77.007203,0.025257,0.038403,7.994688,76.986182,0.012108,0.017382
4,0.032018,307.5,16.1,6,307,0,01-14 23:30,8.03598,76.90095,61e9f38eb937134a3c4bfd8b,...,18.941944,72.885278,8.062509,76.939018,0.026529,0.038068,8.049034,76.919652,0.013054,0.018702


In [18]:
test_df.head()

,ID,vesselId,time,scaling_factor,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,day_of_the_week,hour_of_the_day,...,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day,port_lat,port_long
4,4,61e9f38eb937134a3c4bfd8d,0.349750,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,1,1,0,0,0,0,0,0,48.380556,-4.474167
201,201,61e9f38eb937134a3c4bfd8d,0.349802,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,1,1,1,0,0,0,0,0,48.380556,-4.474167
583,583,61e9f38eb937134a3c4bfd8d,0.349904,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,1,0,0,0,0,0,0,0,48.380556,-4.474167
701,701,61e9f38eb937134a3c4bfd8d,0.349938,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,1,0,0,0,0,0,0,0,48.380556,-4.474167
829,829,61e9f38eb937134a3c4bfd8d,0.349961,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.086957,...,1,1,0,0,0,0,0,0,48.380556,-4.474167


In [19]:
train_df.to_csv('data/processed_data/train.csv', index=False)

test_df.to_csv('data/processed_data/test.csv', index=False)
